# OpenProject Data Sync Notebook

Welcome! This notebook implements the synchronization logic described in `AGENTS.md`. It allows you to parse a local CSV file and create/update Work Packages in OpenProject using the REST API.

## 1. Setup Dependencies

We use the following libraries:
- **Ktor**: For HTTP requests to OpenProject.
- **Kotlin-CSV**: For parsing the source data.
- **Dotenv**: For loading environment variables from `.env` files.

In [ ]:
@file:DependsOn("io.ktor:ktor-client-cio:2.3.12")
@file:DependsOn("io.ktor:ktor-client-content-negotiation:2.3.12")
@file:DependsOn("io.ktor:ktor-serialization-kotlinx-json:2.3.12")
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.10.0")
@file:DependsOn("io.github.cdimascio:dotenv-kotlin:6.4.1")

import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.contentnegotiation.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import io.ktor.serialization.kotlinx.json.*
import kotlinx.serialization.json.*
import com.github.doyaaaaaken.kotlincsv.client.CsvReader
import io.github.cdimascio.dotenv.dotenv
import java.util.Base64

## 2. Load Configuration

We load the credentials and file paths from `.openproject/.env`.

In [ ]:
val dotenv = dotenv {
    directory = ".openproject"
    filename = ".env"
}

val host = dotenv["OPENPROJECT_HOST"]
val apiKey = dotenv["OPENPROJECT_API_KEY"]
val csvPath = dotenv["CSV_FILE_PATH"]

println("Host: $host")
println("CSV Path: $csvPath")

## 3. Define OpenProject Client

This client handles authentication and the POST/PATCH requests.

In [ ]:
val client = HttpClient(CIO) {
    install(ContentNegotiation) {
        json(Json {
            ignoreUnknownKeys = true
        })
    }
}

val encodedAuth = Base64.getEncoder().encodeToString("apikey:$apiKey".toByteArray())

suspend fun createWorkPackage(projectId: String, subject: String, description: String) {
    val response = client.post("$host/api/v3/work_packages") {
        header(HttpHeaders.Authorization, "Basic $encodedAuth")
        contentType(ContentType.Application.Json)
        setBody(buildJsonObject {
            put("subject", subject)
            putJsonObject("description") {
                put("format", "markdown")
                put("raw", description)
            }
            putJsonObject("_links") {
                putJsonObject("project") {
                    put("href", "/api/v3/projects/$projectId")
                }
            }
        })
    }
    
    if (response.status.isSuccess()) {
        println("Successfully created: $subject")
    } else {
        println("Failed to create $subject: ${response.status}")
        println(response.bodyAsText())
    }
}

## 4. Run Sync

Now we parse the CSV and trigger the sync logic.

In [ ]:
import java.io.File

val file = File(csvPath)
if (!file.exists()) {
    println("Error: CSV file not found at $csvPath")
} else {
    val rows = CsvReader().readAllWithHeader(file)
    println("Read ${rows.size} rows from CSV.")

    // Assuming columns: 'project_id', 'subject', 'description'
    rows.forEach { row ->
        val pid = row["project_id"] ?: "1" // Default to 1 if not specified
        val subj = row["subject"] ?: "No Subject"
        val desc = row["description"] ?: ""
        
        createWorkPackage(pid, subj, desc)
    }
}

// Clean up
// client.close() // Usually keep open in notebook for multiple runs